# How to run queries on a model with numpy dataloader

In [1]:
import torch
from agatha.ml.hypothesis_predictor import HypothesisPredictor
from agatha.util.matrix_lookup import np_emb_lookup_table, np_graph
from scipy import sparse
import json

## Initializing a model

In [2]:
model_dir = '/work/acslab/shared/Agatha_data/models_data/2021_11_22_reg_np_dl'

model_path  = f'{model_dir}/model_reg_epoch_7.cpkt'
graph_path  = f'{model_dir}/21_11_22_predicate_graph_csr.npz'
emb_path    = f'{model_dir}/2021_11_22_predicate_embeddings_512.npy'
ent_db_path = f'{model_dir}/21_11_22_predicate_entities_nodelbl_to_int_id_dict.json'

In [4]:
# This flag determines whether or not to load embeddings into RAM.
# It takes a couple minutes upfront (and around 50GB of RAM),
# but large-scale inference will be faster if set to True.

load_emb_into_ram = False

In [5]:
model = HypothesisPredictor.load_from_checkpoint(model_path)

nodelbl_to_int_id_dict = json.load(open(ent_db_path))

emb_np = np_emb_lookup_table(
    nodelbl_to_int_id_dict,
    emb_path,
    memmap=not load_emb_into_ram,
)

graph_csr = sparse.load_npz(graph_path)
graph_db = np_graph(nodelbl_to_int_id_dict, graph_csr)

model.embeddings = emb_np
model.graph = graph_db
model = model.eval()

Memmap is used, embeddings are NOT loaded to RAM.


## Running queries

In [8]:
model.predict_from_terms([("C0006826", "C0040329")])

[0.9849441647529602]